In [1]:
import pandas as pd
import numpy as np

In [28]:
data =pd.read_csv("synthetic_data_2025-12-01 (1).csv")
data

,member_id,claim_id,sex,region,age,consultancy_fee,service_fee,expenses,smoker,procedure_id,total_charges
0,m53147,c35531,male,Southwest,29,134.06,184.88,96.65,yes,71045,415.59
1,m18477,c204276,male,Southwest,76,128.31,197.83,90.26,yes,74019,416.40
2,m36896,c126185,female,Northeast,72,141.87,206.52,91.93,no,82947,440.32
3,m33535,c224089,others,Southeast,42,148.80,206.97,99.02,yes,82947,454.79
4,m20959,c153484,male,Northeast,76,148.98,180.28,100.33,no,29881,429.59
...,...,...,...,...,...,...,...,...,...,...,...
857082,m123012,c133673,male,Northwest,33,145.04,177.54,104.32,yes,85025,426.90
857083,m57849,c87733,others,Northeast,68,146.23,217.34,97.48,yes,29881,461.05
857084,m3302,c157104,female,Southwest,76,132.32,197.94,90.49,no,99213,420.75
857085,m117236,c53373,male,Northwest,42,151.24,205.29,90.59,no,45380,447.12


In [29]:

X = data[['sex', 'region', 'age', 'smoker', 'procedure_id']]
y = data[['consultancy_fee', 'service_fee', 'expenses']]


In [30]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [31]:

categorical_cols = ['sex', 'region', 'smoker', 'procedure_id']
encoder = OneHotEncoder(handle_unknown='ignore')
X_encoded = encoder.fit_transform(X[categorical_cols]).toarray()


scaler = StandardScaler()
scaler.fit(X[['age']])

age_scaled = scaler.transform(X[['age']])
X_final = np.hstack([X_encoded, age_scaled])


In [32]:
X_final

array([[ 0.        ,  1.        ,  0.        , ...,  0.        ,
         0.        , -1.10005257],
       [ 0.        ,  1.        ,  0.        , ...,  0.        ,
         0.        ,  1.48795464],
       [ 1.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  1.2676987 ],
       ...,
       [ 1.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  1.48795464],
       [ 0.        ,  1.        ,  0.        , ...,  0.        ,
         0.        , -0.38422079],
       [ 1.        ,  0.        ,  0.        , ...,  0.        ,
         0.        , -0.98992461]])

In [14]:
data

,member_id,claim_id,sex,region,age,consultancy_fee,service_fee,expenses,smoker,procedure_id,total_charges
0,m53147,c35531,male,Southwest,29,134.06,184.88,96.65,yes,71045,415.59
1,m18477,c204276,male,Southwest,76,128.31,197.83,90.26,yes,74019,416.40
2,m36896,c126185,female,Northeast,72,141.87,206.52,91.93,no,82947,440.32
3,m33535,c224089,others,Southeast,42,148.80,206.97,99.02,yes,82947,454.79
4,m20959,c153484,male,Northeast,76,148.98,180.28,100.33,no,29881,429.59
...,...,...,...,...,...,...,...,...,...,...,...
857082,m123012,c133673,male,Northwest,33,145.04,177.54,104.32,yes,85025,426.90
857083,m57849,c87733,others,Northeast,68,146.23,217.34,97.48,yes,29881,461.05
857084,m3302,c157104,female,Southwest,76,132.32,197.94,90.49,no,99213,420.75
857085,m117236,c53373,male,Northwest,42,151.24,205.29,90.59,no,45380,447.12


In [33]:

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error


In [34]:
X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.2, random_state=42)

In [35]:

model = LinearRegression()
model.fit(X_train, y_train)


LinearRegression()

In [36]:

y_pred = model.predict(X_test)

r2_scores = r2_score(y_test, y_pred, multioutput='raw_values')
rmse_scores = np.sqrt(mean_squared_error(y_test, y_pred, multioutput='raw_values'))

print("R² Scores:", r2_scores)
print("RMSE Scores:", rmse_scores)


R² Scores: [0.63860238 0.4971309  0.62706038]
RMSE Scores: [ 8.39640719 11.28396079  5.72622326]


In [37]:

def predict_fees(sex, region, age, smoker, procedure_id, model):
    
    # Prepare categorical input as list
    categorical_input = [[sex, region, smoker, procedure_id]]

    # Encode categorical features
    encoder = OneHotEncoder(handle_unknown='ignore')
    encoded = encoder.transform(categorical_input).toarray()

    # Scale numeric feature (age)
    age_scaled = StandardScaler().fit_transform([[age]])

    # Combine encoded categorical + scaled numeric
    final_input = np.hstack([encoded, age_scaled])

    # Predict
    prediction = model.predict(final_input)[0]

    return {
        'consultancy_fee': round(prediction[0], 2),
        'service_fee': round(prediction[1], 2),
        'expenses': round(prediction[2], 2)
        
    }


In [43]:

def predict_fees(sex, region, age, smoker, procedure_id):

    categorical_input = [[sex, region, smoker, procedure_id]]

    encoded = encoder.transform(categorical_input).toarray()   
    age_scaled = scaler.transform([[age]])                     

 
    final_input = np.hstack([encoded, age_scaled])

    prediction = model.predict(final_input)[0]

    return [{
        'consultancy_fee': round(prediction[0], 2),
        'service_fee': round(prediction[1], 2),
        'expenses': round(prediction[2], 2)
    },round(prediction[0], 2) + round(prediction[1], 2)+round(prediction[2], 2)]


In [49]:
re = predict_fees('male','Southwest',40,'yes',71045)
print(re[0])
print("Total charges = " + f"{re[1]:.2f}")

{'consultancy_fee': 130.18, 'service_fee': 179.94, 'expenses': 89.97}
Total charges = 400.09


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\base.py:464: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\base.py:464: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
